1. Install Required Libraries
pip install -r requirements.txt

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../..")

2. Azure Blob Setup

In [3]:
from src.config import AZURE_CONNECTION_STRING, CONTAINER_TRAIN_MOMENTS
from azure.storage.blob import BlobServiceClient

blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_TRAIN_MOMENTS)

3. Import Core Libraries

In [4]:
from confluent_kafka import Consumer
import json
import pandas as pd
from lxml import etree
from datetime import datetime
from io import StringIO

4. Azure Upload Function

In [5]:
def upload_to_azure(records, blob_name):
    df = pd.DataFrame(records)

    buffer = StringIO()
    df.to_csv(buffer, index=False)

    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(buffer.getvalue(), overwrite=True)

    print(f"Uploaded {len(records)} records → {blob_name}")

5. Batch + File Naming Logic

In [7]:
BATCH_SIZE = 100
UPLOAD_SIZE = 2000

def generate_blob_name():
    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    return f"train_data_{timestamp}.csv"

6. Parse train details

In [8]:
def parse_train_batch(data_list):
    records = []

    for item in data_list:
        header = item.get("header", {})
        body = item.get("body", {})
        # print(body)

        record = {
            "train_id": body.get("train_id"),

            # Time
            "actual_timestamp": body.get("actual_timestamp"),
            "planned_timestamp": body.get("planned_timestamp"),
            "gbtt_timestamp": body.get("gbtt_timestamp"),

            # Location
            "loc_stanox": body.get("loc_stanox"),
            "next_stanox": body.get("next_report_stanox"),

            # Event
            "event_type": body.get("event_type"),
            "planned_event_type": body.get("planned_event_type"),

            # Delay
            "variation_status": body.get("variation_status"),
            "timetable_variation": body.get("timetable_variation"),

            # Operational
            "platform": str(body.get("platform")).strip() if body.get("platform") else None,
            "route": body.get("route"),
            "direction": body.get("direction_ind"),

            # Flags
            "train_terminated": body.get("train_terminated"),
            "delay_monitoring_point": body.get("delay_monitoring_point"),

            # Header
            "msg_timestamp": header.get("msg_queue_timestamp"),
            "data_source": header.get("original_data_source")
        }

        # Derived feature
        record["is_delayed"] = 1 if record["variation_status"] == "LATE" else 0

        records.append(record)

    return records

7. Config and Main Pipleine

In [9]:
# =========================
# CONFIG
# =========================
from src.config import TRAIN_MOMENTS_CONFIG, TRAIN_MOMENTS_TOPIC

consumer = Consumer(TRAIN_MOMENTS_CONFIG)
consumer.subscribe([TRAIN_MOMENTS_TOPIC])

print("Streaming NWR Train Moments data → Azure...")

all_records = []
chunk_counter = 0
current_blob = generate_blob_name()


try:
    while True:
        msg = consumer.poll(1.0)

        if msg is None:

            continue

        if msg.error():
            print("Error:", msg.error())
            continue

        try:
            # Step 1: Decode Kafka message
            raw_str = msg.value().decode('utf-8')
            data = json.loads(raw_str)

            parsed = parse_train_batch(data)

            all_records.extend(parsed)

            # Batch upload
            if len(all_records) >= BATCH_SIZE:
                upload_to_azure(all_records, current_blob)

                chunk_counter += len(all_records)
                all_records.clear()

                # File rotation
                if chunk_counter >= UPLOAD_SIZE:
                    print("Chunk limit reached → rotating file")
                    current_blob = generate_blob_name()
                    chunk_counter = 0


        except Exception as e:
            print("Processing error:", e)
            continue   # keep stream running

except KeyboardInterrupt:
    print("Stopping stream...")

finally:
    upload_to_azure(all_records, current_blob)
    print(f"Saved {len(all_records)} records")


    consumer.close()
    print("Consumer closed")


Streaming NWR Train Moments data → Azure...
Uploaded 103 records → train_data_20260426_151337.csv
Uploaded 127 records → train_data_20260426_151337.csv
Uploaded 128 records → train_data_20260426_151337.csv
Uploaded 117 records → train_data_20260426_151337.csv
Uploaded 102 records → train_data_20260426_151337.csv
Uploaded 103 records → train_data_20260426_151337.csv
Uploaded 118 records → train_data_20260426_151337.csv
Uploaded 123 records → train_data_20260426_151337.csv
Uploaded 114 records → train_data_20260426_151337.csv
Uploaded 119 records → train_data_20260426_151337.csv
Uploaded 107 records → train_data_20260426_151337.csv
Uploaded 105 records → train_data_20260426_151337.csv
Uploaded 109 records → train_data_20260426_151337.csv
Uploaded 123 records → train_data_20260426_151337.csv
Uploaded 108 records → train_data_20260426_151337.csv
Stopping stream...
Uploaded 8 records → train_data_20260426_151337.csv
Saved 8 records
Consumer closed
